# ML In-Class Exam — Cheatsheet
### Binary Classification with scikit-learn

**Example dataset used throughout:** A fictional bank loan dataset — 900 customer records, predicting whether a customer will default on their loan (`Defaulted`: 0 = No, 1 = Yes).

Columns: `CustomerID`, `Name`, `Age`, `Income`, `LoanAmount`, `CreditScore`, `EmploymentYears`, `DebtToIncomeRatio`, `Defaulted`

> Replace the example column names, filename, and student ID with your actual values when the exam starts.

In [ ]:
# Run this first — all imports needed for the entire notebook
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

np.random.seed(419414)  # change to your student ID

---
## Exploratory Data Analysis

In [ ]:
# Load the dataset and display shape and first few rows
df = pd.read_csv('loans.csv')  # change to your filename

print('Dataset shape:', df.shape)
# Example output: (900, 9) — 900 rows, 9 columns

df.head()
# Example output:
#   CustomerID  Name   Age  Income   LoanAmount  CreditScore  EmploymentYears  DebtToIncomeRatio  Defaulted
#   1001        Alice  34   52000    15000       680          5                0.31               0
#   1002        Bob    27   38000    22000       590          2                0.58               1

In [ ]:
# Check column data types and non-null counts
df.info()
# Example output:
# RangeIndex: 900 entries
# CustomerID          900 non-null  int64
# Name                900 non-null  object
# Age                 900 non-null  int64
# Income              900 non-null  float64
# LoanAmount          900 non-null  float64
# CreditScore         900 non-null  int64
# EmploymentYears     900 non-null  int64
# DebtToIncomeRatio   900 non-null  float64
# Defaulted           900 non-null  int64

In [ ]:
# Explicit null count per column
df.isnull().sum()
# Example output — all zeros means no nulls:
# CustomerID          0
# Age                 0
# CreditScore         0  <- but check describe() for suspicious zero values below

In [ ]:
# Statistical summary — useful for spotting suspicious min values of 0
df.describe()
# Example: if CreditScore min = 0, that is not a valid score and likely missing data
# If a column that should never be 0 has a min of 0, treat it in preprocessing

In [ ]:
# Plot the class distribution of the target column
sns.countplot(data=df, x='Defaulted')  # change 'Defaulted' to your target column
plt.title('Class Distribution of Defaulted')
plt.show()

print(df['Defaulted'].value_counts())
# Example output:
# 0    765   <- did not default (majority class)
# 1    135   <- defaulted (minority class)
# Only ~15% positive cases — dataset is imbalanced

In [ ]:
# Boxplots for at least 4 features split by target class
# Wide, non-overlapping boxes = good predictor
# Heavily overlapping boxes = less useful feature
features = ['CreditScore', 'DebtToIncomeRatio', 'Income', 'LoanAmount']  # change to your columns

plt.figure(figsize=(12, 8))
for i, col in enumerate(features, 1):
    plt.subplot(2, 2, i)
    sns.boxplot(data=df, x='Defaulted', y=col)  # change 'Defaulted' to your target
    plt.title(f'{col} by Defaulted')
plt.tight_layout()
plt.show()

# Example interpretation:
# CreditScore       -> defaulters have much lower scores (clear separation = informative)
# DebtToIncomeRatio -> defaulters have higher ratios (good separation)
# Income            -> some overlap but moderate separation
# LoanAmount        -> lots of overlap, less discriminative on its own

### EDA Summary

The dataset contains 900 customer records with a moderately imbalanced class distribution — 765 non-defaulters (85%) vs 135 defaulters (15%). No explicit null values were found, however `CreditScore` contains a small number of zeros which are not a valid credit score value and likely represent missing data encoded as zero, requiring treatment before modelling. From the visualisations, `CreditScore` and `DebtToIncomeRatio` show the clearest separation between the two classes, suggesting they will be the most informative predictors. `LoanAmount` shows substantial overlap between classes and appears less individually discriminative. Overall the dataset is usable but requires imputation of invalid zero values and removal of identifier columns before modelling.

---
## Data Preparation

### Data Preparation Justification

**Option A — use when zeros are not valid (e.g. CreditScore = 0 is impossible):**
Columns `CreditScore` and `Income` contained zeros that are not plausible real-world values and were replaced with NaN. Median imputation was applied because these features are skewed, and the median is more robust to outliers than the mean. This preserves the full dataset size while correcting the invalid entries.

**Option B — use when dropping identifier or string columns:**
`CustomerID` and `Name` were dropped before modelling — `CustomerID` is an arbitrary identifier with no predictive value, and `Name` is a free-text string that cannot be used as a numeric feature. The remaining features are all numeric and require no further imputation.

In [ ]:
# OPTION A — Replace invalid zeros with NaN then impute with median
# Use when a column cannot logically be zero (e.g. CreditScore, Income, BloodPressure)
invalid_zero_cols = ['CreditScore', 'Income']  # change to your columns

df[invalid_zero_cols] = df[invalid_zero_cols].replace(0, np.nan)

imputer = SimpleImputer(strategy='median')  # median is robust to skewed data
df[invalid_zero_cols] = imputer.fit_transform(df[invalid_zero_cols])

In [ ]:
# OPTION B — Drop columns that are not usable as features
# Use when dataset has IDs, names, or high-cardinality string columns
df = df.drop(columns=['CustomerID', 'Name'])  # change to your columns

In [ ]:
# Separate features (X) and target (y), then do 60/20/20 stratified split
X = df.drop('Defaulted', axis=1)   # change 'Defaulted' to your target column
y = df['Defaulted'].astype(int)    # .astype(int) converts True/False booleans to 1/0

student_id = 419414  # change to your student ID

# 60% train, 40% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, stratify=y, random_state=student_id
)

# Split temp 50/50 -> 20% val, 20% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=student_id
)

X_train.shape, X_val.shape, X_test.shape
# Example output: (540, 6), (180, 6), (180, 6)

In [ ]:
# Scale features — fit ONLY on training data, then apply to val and test
# Fitting on val/test too would be data leakage
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

---
## Model Training — Model A: Logistic Regression

#### Why Logistic Regression?
Logistic Regression is a strong baseline choice because `CreditScore` and `DebtToIncomeRatio` show clear linear separation between defaulters and non-defaulters in the boxplots. It is interpretable, computationally efficient, and performs well when features are scaled — which has already been applied. It also provides probabilistic outputs, which is useful when dealing with imbalanced classes where understanding prediction confidence matters.

In [ ]:
# Logistic Regression with GridSearchCV — tuning C (regularisation) and solver
log_reg = LogisticRegression(max_iter=1000)

param_grid_lr = {
    'C': [0.01, 0.1, 1, 10],           # smaller C = stronger regularisation
    'solver': ['liblinear', 'lbfgs']    # optimisation algorithm
}

grid_lr = GridSearchCV(
    estimator=log_reg,
    param_grid=param_grid_lr,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_lr.fit(X_train_scaled, y_train)

In [ ]:
print('Model A — Logistic Regression')
print('Best Parameters:', grid_lr.best_params_)
print('Mean CV Score:  ', round(grid_lr.best_score_, 4))
# Example output:
# Best Parameters: {'C': 0.1, 'solver': 'liblinear'}
# Mean CV Score:   0.9222

---
## Model Training — Model B: k-Nearest Neighbours

#### Why KNN?
KNN is a non-parametric classifier that makes no assumptions about the underlying data distribution, allowing it to capture non-linear decision boundaries. This is useful here as the relationship between features like `EmploymentYears` and default risk may not be strictly linear. KNN benefits directly from the feature scaling already applied, and tuning the number of neighbours and distance metric allows it to adapt well to the size and characteristics of the dataset.

In [ ]:
# KNN with GridSearchCV — tuning n_neighbors, weights, and distance metric
knn = KNeighborsClassifier()

param_grid_knn = {
    'n_neighbors': [3, 5, 7, 9, 11],       # how many neighbours to consider
    'weights': ['uniform', 'distance'],     # uniform = all equal, distance = closer = more weight
    'metric': ['euclidean', 'manhattan']    # how distance between points is measured
}

grid_knn = GridSearchCV(
    estimator=knn,
    param_grid=param_grid_knn,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_knn.fit(X_train_scaled, y_train)

In [ ]:
print('Model B — KNN')
print('Best Parameters:', grid_knn.best_params_)
print('Mean CV Score:  ', round(grid_knn.best_score_, 4))
# Example output:
# Best Parameters: {'metric': 'manhattan', 'n_neighbors': 7, 'weights': 'distance'}
# Mean CV Score:   0.9315

---
## Model Training — Model C: Random Forest

#### Why Random Forest?
Random Forest handles non-linear relationships and naturally models interactions between features — for example, the combination of high `DebtToIncomeRatio` and low `CreditScore` may jointly predict default better than either feature alone. It is robust to noise, provides built-in feature importance, and unlike Logistic Regression and KNN does not require feature scaling. Averaging predictions across many decorrelated trees reduces overfitting risk, making it well-suited to datasets with class imbalance.

In [ ]:
# Random Forest with GridSearchCV — tuning n_estimators, max_depth, min_samples_split
# IMPORTANT: Random Forest does NOT need scaled data — use X_train not X_train_scaled
rf = RandomForestClassifier(random_state=student_id)

param_grid_rf = {
    'n_estimators': [100, 200, 300],       # number of trees in the forest
    'max_depth': [None, 5, 10, 20],        # max depth per tree (None = no limit)
    'min_samples_split': [2, 5, 10]        # min samples needed to split a node
}

grid_rf = GridSearchCV(
    estimator=rf,
    param_grid=param_grid_rf,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_rf.fit(X_train, y_train)  # unscaled data for RF

In [ ]:
print('Model C — Random Forest')
print('Best Parameters:', grid_rf.best_params_)
print('Mean CV Score:  ', round(grid_rf.best_score_, 4))
# Example output:
# Best Parameters: {'max_depth': 10, 'min_samples_split': 2, 'n_estimators': 200}
# Mean CV Score:   0.9463

---
## Model Comparison

In [ ]:
# Evaluate all three best models on the validation set
# LR and KNN use scaled data — RF uses unscaled
best_lr  = grid_lr.best_estimator_
best_knn = grid_knn.best_estimator_
best_rf  = grid_rf.best_estimator_

pred_lr  = best_lr.predict(X_val_scaled)   # scaled
pred_knn = best_knn.predict(X_val_scaled)  # scaled
pred_rf  = best_rf.predict(X_val)          # unscaled

results = {
    'Model': ['Logistic Regression', 'KNN', 'Random Forest'],
    'Accuracy': [
        accuracy_score(y_val, pred_lr),
        accuracy_score(y_val, pred_knn),
        accuracy_score(y_val, pred_rf)
    ],
    'F1 Score': [
        f1_score(y_val, pred_lr),
        f1_score(y_val, pred_knn),
        f1_score(y_val, pred_rf)
    ]
}

In [ ]:
results_df = pd.DataFrame(results)
results_df
# Example output:
#                  Model  Accuracy  F1 Score
# 0  Logistic Regression    0.9167    0.6154
# 1                  KNN    0.9278    0.6667
# 2        Random Forest    0.9444    0.7500  <- highest on both metrics

### Model Comparison Summary

Random Forest outperforms both Logistic Regression and KNN on the validation set, achieving the highest accuracy and F1 score. This is likely because it models non-linear interactions between features such as `CreditScore` and `DebtToIncomeRatio` more effectively than a linear classifier, and its ensemble approach makes it more robust to noise in features like `LoanAmount`. Given the class imbalance in this dataset (765 non-defaulters vs 135 defaulters), F1 score is the more informative metric — a model that predicted 'no default' for every customer would achieve approximately 85% accuracy while identifying zero actual defaulters, making accuracy alone very misleading. F1 score accounts for both false positives and false negatives, making it a much better indicator of whether the model is genuinely learning to detect the minority class. Logistic Regression performs reasonably well but its lower F1 score suggests it struggles to correctly identify defaulters due to the class imbalance and potential non-linearity in the data.

---
## Final Evaluation

### Final Model Selection

I am selecting the Random Forest classifier as my final model. It achieved the highest accuracy and F1 score on the validation set, demonstrating better generalisation than Logistic Regression and KNN. Its ability to model non-linear feature interactions and handle class imbalance without requiring feature scaling makes it the most appropriate choice for this dataset.

In [ ]:
# Retrain final model on combined train + validation data using best hyperparameters
# Gives the model more training data before final test set evaluation
X_train_final = np.vstack([X_train, X_val])   # for LR/KNN use: X_train_scaled + X_val_scaled
y_train_final = np.hstack([y_train, y_val])

final_rf = RandomForestClassifier(
    **grid_rf.best_params_,   # unpacks best params found by GridSearchCV
    random_state=student_id
)

final_rf.fit(X_train_final, y_train_final)

In [ ]:
# Evaluate on the held-out test set and print the full classification report
test_pred = final_rf.predict(X_test)  # for LR/KNN use: X_test_scaled

print('Final Model — Random Forest')
print(classification_report(y_test, test_pred))
# Example output:
#               precision  recall  f1-score  support
#            0      0.97    0.99      0.98      153   <- non-defaulters: near perfect
#            1      0.93    0.81      0.87       27   <- defaulters: lower recall, missing some
#     accuracy                        0.96      180

In [ ]:
# Confusion matrix heatmap
# Top-left:     true negatives  (correctly predicted no default)
# Top-right:    false positives (predicted default, actually fine)
# Bottom-left:  false negatives (missed actual defaulters) <- most costly error
# Bottom-right: true positives  (correctly predicted default)
cm = confusion_matrix(y_test, test_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Default', 'Default'],
            yticklabels=['No Default', 'Default'])
plt.title('Confusion Matrix — Final Random Forest Model')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

### Final Model Interpretation

The final model performs very well on the non-defaulting class, achieving near-perfect precision and recall, but shows a drop in recall for actual defaulters where a number of true positive cases are missed. This asymmetry is a direct consequence of class imbalance — with only 27 defaulters in the test set, the model has had limited exposure to positive examples during training, making it harder to generalise to all defaulter patterns. The confusion matrix confirms this, showing errors are concentrated in false negatives on the minority class rather than false positives. While the overall accuracy is strong, deploying this model in a real financial setting would require careful consideration — missing a genuine defaulter carries a significant cost, and techniques such as lowering the classification threshold, applying SMOTE oversampling, or using `class_weight='balanced'` should be explored to improve minority class recall before any real-world deployment.